In [ ]:
#Import libraries (ipyleaflet, geopandas, ipywidgets, rest are dependencies)
from ipyleaflet import Map, Marker, GeomanDrawControl, FullScreenControl, ScaleControl, WidgetControl, GeoData
import geopandas as gpd
import pandas as pd
from shapely.geometry import shape
import ipywidgets as widgets
import os
import io

In [ ]:
# Clears cache, upload other layers
filecache = []
upload = widgets.FileUpload(accept=".gpkg,.geojson", multiple=True)
upload

In [ ]:
# Loads file into cache, repeat if files are in different folders (wait a second after selecting file to load it in)
filecache.extend(upload.value)
[filecache[i]["name"] for i in range(len(filecache))]

In [ ]:
# Reset Map and overlay, modify map details here
m = Map(center=(38,-96), zoom=5, scroll_wheel_zoom=True, 
        layout={"height": "900px", "width": "100%"})

# Select colors for each file; pass None to not display; or "active:Color" to set as the active with Color ()  
uploadcolors = ["active:teal"]
uploadopacity = 0.6
uploadweight = 6

In [ ]:
# Overlay code and display
newtable = gpd.GeoDataFrame()

for i in range(len(filecache)):
    if not uploadcolors[i]:
        continue
    elif uploadcolors[i][:6].lower() == "active":
        activefile = filecache[i]
        newtable = gpd.read_file(io.BytesIO(filecache[i].content))
        m.add(GeoData(geo_dataframe=newtable, style={"color": uploadcolors[i][7:], "opacity": uploadopacity, "weight": uploadweight, "fillOpacity": uploadopacity/2},
                point_style={"radius": 4, "color": uploadcolors[i][7:], "fillOpacity": uploadopacity/2, "weight": uploadweight},))
    else:
        geod = gpd.read_file(io.BytesIO(filecache[i].content))
        layername = filecache[i]["name"].split(".")[0]
        m.add(GeoData(geo_dataframe=geod, name=layername, style={"color": uploadcolors[i], "opacity": uploadopacity, "weight": uploadweight, "fillOpacity": uploadopacity/2},
                    point_style={"radius": 4, "color": uploadcolors[i], "fillOpacity": uploadopacity/2, "weight": uploadweight},))


m.add(ScaleControl(max_width=200, position='bottomleft'))

draw_control = GeomanDrawControl(edit=False, drag=False, cut=False, rotate=False)
def draw_update(target, action, geo_json):
    global newtable
    if action == "create":
        camselection = [cam for cam, index in zip(cams, range(len(cams))) if butlist[index].button_style=="success"]
        newtable = pd.concat([newtable, gpd.GeoDataFrame({"imagedate":[imagedate.value],"recorddate":[recorddate.value],"tags":[str(tags.value)],
        "camselection":[str(camselection)],"geometry":[shape(geo_json[0]["geometry"])]})], ignore_index=True)
    elif action == "remove":
        newtable = newtable.drop(newtable[newtable.geometry==shape(geo_json[0]["geometry"])].index)
    pass
def color_update(_erm):
    draw_control.circlemarker = {
        "pathOptions": {
            "fillColor": fillcolor.value,
            "color": linecolor.value,
            "weight": 5,
            "fillOpacity": 0.4
        },
        "snappable": False,
    }
    draw_control.polyline =  {
        "pathOptions": {
            "color": linecolor.value,
            "weight": 5,
            "opacity": 0.7
        }
    }
    draw_control.polygon = {
        "pathOptions": {
            "fillColor": fillcolor.value,
            "color": linecolor.value,
            "fillOpacity": 0.5
        }
    }
draw_control.on_draw(draw_update)

picturefolder = "./cams/"
wd = os.fsencode(picturefolder)
filec = len(os.listdir(wd))
cams = []
camimages = []
for file in sorted(os.listdir(wd)):
    filename = os.fsdecode(file)
    cams.append(filename.split(".")[0])
    camimages.append(filename)
imgframe = widgets.Image(value=open(picturefolder + camimages[0],"rb").read(), width=300, height=300)

def changeimg(img, buttonnum):
    imgframe.value = open(picturefolder + img, "rb").read()
    if butlist[buttonnum].button_style == "success":
        butlist[buttonnum].button_style = "danger"
    else:
        butlist[buttonnum].button_style = "success"
    return

butlist = [widgets.Button(description=cam, button_style="danger") for cam in cams]
[button.on_click(lambda x, img=camimage, cnt=count: changeimg(img, cnt)) for button, camimage, count in zip(butlist, camimages, range(len(camimages)))]
camwidget = widgets.HBox([widgets.VBox(butlist), imgframe])


imagedate = widgets.DatePicker(description="image_date")
recorddate = widgets.DatePicker(description="record_date")
tags = widgets.TagsInput(value=["Tags"], allow_duplicates=False)

accordion = widgets.Accordion(children=[widgets.VBox([imagedate, recorddate, tags]),camwidget], titles=("Metadata", "Cams"), layout=widgets.Layout(max_width="500px"))

linecolor = widgets.ColorPicker(description="Line Color")
fillcolor = widgets.ColorPicker(description="Fill Color")
linecolor.observe(color_update)
fillcolor.observe(color_update)

brdatawidget = WidgetControl(widget=accordion, position="bottomright", layout=widgets.Layout(max_width="50%"))
trcolorwidget = WidgetControl(widget=widgets.VBox([linecolor, fillcolor]), position="topright")
color_update(None)
m.add(brdatawidget)
m.add(trcolorwidget)
m.add(draw_control)


display(m)

In [ ]:
# To save the modified table we have to write to the file, the else statement is for when an active file is not selected (ie; change "filenamegoeshere.gpkg").
# Filepath needs to be added otherwise it will save in the same folder
if activefile:
    newtable.to_file("./"+activefile["name"])
else:
    newtable.to_file("filenamegoeshere.gpkg")


Note: The file has to be reselected and put into cache for the saved changes to be represented.
